# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AshenDary/StarterRepo/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I am choosing Lane 2: Refresh / Content Opportunity Scoring. This lane fits the capstone because the useful product question is not "can I guarantee that a refresh will work?" but "which pages should a content team review first when time is limited?" The output should support a human reviewer by ranking pages with visible evidence of decline, weakness, or opportunity, while staying honest that the data is observational and cannot prove that editing a page will cause recovery.

In [3]:
# No numeric claims in this section.
# The lane choice and framing are drafted in the markdown cell above.


## 2. The question: decision, action, cost of a wrong call

This project improves the decision of which content pages a reviewer should inspect first for a possible refresh or other content action. A content strategist, editor, or SEO reviewer would use the ranked queue to decide whether to refresh, expand, protect, prune, or monitor a page after checking the evidence and business context. A wrong call has two main costs: a false positive wastes reviewer time on a page that did not need attention, while a false negative lets a genuinely declining or underperforming page stay buried in the backlog.

In [4]:
# No numeric claims in this section.
# The decision, action, and cost framing are drafted in the markdown cell above.


## 3. Quick look at the data (2-3 real numbers)

The cell below loads the starter CSV and computes a few aggregate checks from documented columns only. These numbers are evidence for why a ranked review queue could be useful, not proof that any recommended action will cause recovery.

In [5]:
from csv import DictReader
from pathlib import Path

DATA_FILENAME = "content_refresh_anonymized.csv"
RELATIVE_DATA_PATH = Path("data") / "raw" / DATA_FILENAME


def find_data_file() -> Path:
    """Find the starter CSV whether the notebook runs from repo root or work/notebooks."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / RELATIVE_DATA_PATH
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {RELATIVE_DATA_PATH} from {Path.cwd()}")


def to_number(row: dict[str, str], column: str) -> float:
    value = row.get(column, "")
    return float(value) if value not in ("", None) else 0.0


data_path = find_data_file()
with data_path.open(newline="") as file:
    rows = list(DictReader(file))

required_columns = {
    "content_id",
    "client_id",
    "trend_direction",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "content_age_days",
    "ctr",
}
missing_columns = required_columns - set(rows[0])
if missing_columns:
    raise ValueError(f"Missing expected columns: {sorted(missing_columns)}")

total_pages = len(rows)
client_count = len({row["client_id"] for row in rows})

# Starter guide reason-code logic, computed from observable columns only.
declining_pages = sum(row["trend_direction"] == "down" for row in rows)
declining_with_demand = sum(
    row["trend_direction"] == "down" and to_number(row, "impressions_90d") >= 100
    for row in rows
)
low_ctr_visible_page = sum(
    to_number(row, "impressions_90d") >= 500
    and 0 < to_number(row, "avg_position") <= 20
    and to_number(row, "ctr") < 0.5
    for row in rows
)
page_one_decay_risk = sum(
    0 < to_number(row, "avg_position") <= 10
    and to_number(row, "content_age_days") >= 180
    for row in rows
)


def pct(count: int) -> str:
    return f"{count / total_pages:.1%}"


summary = {
    "starter_rows_pages": total_pages,
    "pseudonymized_clients": client_count,
    "trend_direction_down_pages": f"{declining_pages} ({pct(declining_pages)})",
    "declining_with_demand_pages": f"{declining_with_demand} ({pct(declining_with_demand)})",
    "low_ctr_visible_page_reason_code": f"{low_ctr_visible_page} ({pct(low_ctr_visible_page)})",
    "page_one_decay_risk_reason_code": f"{page_one_decay_risk} ({pct(page_one_decay_risk)})",
}

summary


{'starter_rows_pages': 30000,
 'pseudonymized_clients': 32,
 'trend_direction_down_pages': '16262 (54.2%)',
 'declining_with_demand_pages': '13152 (43.8%)',
 'low_ctr_visible_page_reason_code': '9759 (32.5%)',
 'page_one_decay_risk_reason_code': '7076 (23.6%)'}

## 4. Careful words: what I can and can't claim

- I can say the starter data shows observable patterns in anonymized search, content, and engagement signals that may help prioritize review.
- I can use cautious wording such as "we observed," "this suggests," and "this page may be worth reviewing before lower-scored pages."
- I cannot claim that refreshing a page causes recovery unless I run a valid experiment or causal design.
- I cannot claim that these signals reveal Google's ranking algorithm, and I will not use or justify the project with product-decision fields such as `health_score`, `priority_score`, or `action_type`.

In [6]:
# No numeric claims in this section.
# The allowed and not-allowed claim language is drafted in the markdown cell above.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
